# ABSA Full Pipeline — Fine-tune + Baselines + XAI
**All-in-one notebook** cho demo và báo cáo.

| Bước | Nội dung |
|------|----------|
| 1 | Setup: clone repo, cài thư viện, tìm dataset |
| 2 | Fine-tune Transformer (BERT/RoBERTa/…) |
| 3 | Train Baselines (LSTM / BiLSTM / RNN) |
| 4 | So sánh kết quả các mô hình |
| 5 | XAI: LIME, SHAP, Integrated Gradients, Attention, Gradient-based |
| 6 | Lưu toàn bộ kết quả ra `/kaggle/working/` |

## Bước 1 — Setup

In [ ]:
import subprocess, os, glob

REPO_URL = 'https://github.com/haiyen040602/xai-absa.git'
REPO_DIR = '/kaggle/working/xai-transformer'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo to', REPO_DIR)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo already present, pulled latest.')

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cài thư viện
!pip install -q transformers lime shap captum seaborn scikit-learn

In [ ]:
# Tìm dataset
import glob

def find_dataset():
    # 1. Local (khi chạy offline)
    for local in ['Datasets/FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl',
                  '../Datasets/FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl']:
        if os.path.exists(local):
            return os.path.abspath(local)
    # 2. Kaggle Input
    for pattern in ['/kaggle/input/*/*.jsonl', '/kaggle/input/*/*/*.jsonl']:
        matches = glob.glob(pattern)
        if matches:
            return matches[0]
    return None

DATASET_PATH = find_dataset()
if DATASET_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay dataset JSONL. '
        'Vui long attach dataset vao Kaggle Input hoac dung Add Data.'
    )
print('Dataset:', DATASET_PATH)

## Bước 2 — Fine-tune Transformer

> Thay `MODEL_NAME` để chọn mô hình khác: `roberta-base`, `distilbert-base-uncased`, `albert-base-v2`, `xlnet-base-cased`.

In [ ]:
# ===== CẤU HÌNH =====
MODEL_NAME   = 'bert-base-uncased'   # thay tên model ở đây
EPOCHS       = 3
BATCH_SIZE   = 8
LR           = 1e-5
MAX_LEN      = 128
OUTPUT_DIR   = '/kaggle/working/absa_outputs'
SEED         = 42
# =====================

cmd = [
    'python3', 'kaggle/run_absa_repro.py',
    '--experiment', 'transformer',
    '--model_name', MODEL_NAME,
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--learning_rate', str(LR),
    '--max_length', str(MAX_LEN),
    '--output_dir', OUTPUT_DIR,
    '--seed', str(SEED),
    '--dataset_path', DATASET_PATH,
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
print('Return code:', result.returncode)

## Bước 3 — Train Baselines (LSTM / BiLSTM / RNN)

In [ ]:
# ===== CẤU HÌNH BASELINE =====
BL_EPOCHS   = 10
BL_BATCH    = 8
BL_LR       = 1e-3
BL_MAX_LEN  = 50
# ==============================

cmd_bl = [
    'python3', 'kaggle/run_absa_repro.py',
    '--experiment', 'all',           # all = transformer + lstm + bilstm + rnn
    '--model_name', MODEL_NAME,
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--learning_rate', str(LR),
    '--max_length', str(MAX_LEN),
    '--baseline_epochs', str(BL_EPOCHS),
    '--baseline_batch_size', str(BL_BATCH),
    '--baseline_learning_rate', str(BL_LR),
    '--baseline_max_length', str(BL_MAX_LEN),
    '--output_dir', OUTPUT_DIR,
    '--seed', str(SEED),
    '--dataset_path', DATASET_PATH,
]

print('Running:', ' '.join(cmd_bl))
result_bl = subprocess.run(cmd_bl, capture_output=False, text=True)
print('Return code:', result_bl.returncode)

## Bước 4 — So sánh kết quả các mô hình

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

table_path = os.path.join(OUTPUT_DIR, 'comparison_table.csv')
if not os.path.exists(table_path):
    print('comparison_table.csv chua co. Hay chay buoc 3 (--experiment all) truoc.')
else:
    df = pd.read_csv(table_path)
    display(df)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df))
    width = 0.35
    bars1 = ax.bar([i - width/2 for i in x], df['accuracy'],  width, label='Accuracy',  color='#4C78A8')
    bars2 = ax.bar([i + width/2 for i in x], df['macro_f1'], width, label='Macro F1', color='#F58518')

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(list(x))
    ax.set_xticklabels(df['model'], rotation=15, ha='right')
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Accuracy vs Macro F1')
    ax.legend()
    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, 'comparison_chart.png')
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print('Saved:', chart_path)

In [ ]:
# Confusion matrix subplots (4 models)
import matplotlib.image as mpimg

model_names = ['transformer', 'lstm', 'bilstm', 'rnn']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, mname in zip(axes.flat, model_names):
    img_path = os.path.join(OUTPUT_DIR, mname, 'confusion_matrix.png')
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(mname.upper(), fontsize=14)
    else:
        ax.text(0.5, 0.5, 'Missing file:\n' + img_path, ha='center', va='center', fontsize=9)
        ax.set_title(mname.upper() + ' (not found)', fontsize=12)
    ax.axis('off')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')
plt.tight_layout()
cm_grid_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_grid.png')
plt.savefig(cm_grid_path, dpi=150)
plt.show()
print('Saved:', cm_grid_path)

## Bước 5 — XAI Explanations

Load model fine-tuned tốt nhất, rồi chạy 5 kỹ thuật giải thích.

> Thay `EXPLAIN_TEXT` và `EXPLAIN_ASPECT` để thử với câu và khía cạnh khác.

In [ ]:
import torch, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import shap
from captum.attr import LayerIntegratedGradients

sns.set_theme(style='whitegrid')

# ===== CẤU HÌNH XAI =====
EXPLAIN_TEXT   = 'The food was amazing but the service was very slow.'
EXPLAIN_ASPECT = 'service'
XAI_OUT_DIR    = '/kaggle/working/xai_outputs'
os.makedirs(XAI_OUT_DIR, exist_ok=True)
# =========================

# Tìm model fine-tuned
MODEL_PATH = None
for p in [
    os.path.join(OUTPUT_DIR, 'transformer'),
    *glob.glob('/kaggle/input/*/transformer'),
    *glob.glob('/kaggle/input/*/*transformer*'),
]:
    if os.path.exists(os.path.join(p, 'config.json')):
        MODEL_PATH = p
        break

if MODEL_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay model fine-tuned. '
        'Hay chay Buoc 2 truoc hoac attach model vao Kaggle Input.'
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

name_hint = (model.config._name_or_path or '').lower()
if 'roberta' in name_hint:
    SEP = '</s></s>'
elif 'xlnet' in name_hint:
    SEP = '<sep>'
else:
    SEP = '[SEP]'

label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
print('Model loaded from:', MODEL_PATH)
print('Separator:', SEP, '| Device:', device)

def build_input(t, a): return t + ' ' + SEP + ' ' + a

def predict_proba(texts, aspects=None):
    asp = aspects if aspects else [EXPLAIN_ASPECT] * len(texts)
    probs = []
    for t, a in zip(texts, asp):
        enc = tokenizer(build_input(t, a), return_tensors='pt',
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc).logits
            probs.append(torch.softmax(out, dim=-1).cpu().numpy()[0])
    return np.array(probs)

probs = predict_proba([EXPLAIN_TEXT])[0]
pred = int(np.argmax(probs))
print('\nInput:', build_input(EXPLAIN_TEXT, EXPLAIN_ASPECT))
print('Predicted:', label_map[pred], '| Probs:', np.round(probs, 4))

### 5.1 — LIME

In [ ]:
lime_explainer = LimeTextExplainer(class_names=['negative', 'neutral', 'positive'])

exp = lime_explainer.explain_instance(
    EXPLAIN_TEXT,
    lambda txts: predict_proba(txts),
    num_features=10,
    num_samples=1000
)

fig_lime = exp.as_pyplot_figure(label=pred)
fig_lime.suptitle('LIME — class: ' + label_map[pred])
lime_path = os.path.join(XAI_OUT_DIR, 'lime_explanation.png')
fig_lime.savefig(lime_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', lime_path)

### 5.2 — SHAP

In [ ]:
def shap_predict(raw_texts):
    inputs = [build_input(t, EXPLAIN_ASPECT) for t in raw_texts]
    enc = tokenizer(inputs, return_tensors='pt',
                    truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        return torch.softmax(logits, dim=-1).cpu().numpy()

shap_explainer = shap.Explainer(shap_predict, tokenizer)
shap_values = shap_explainer([EXPLAIN_TEXT])
shap.initjs()
shap.text_plot(shap_values)

### 5.3 — Integrated Gradients (Captum)

In [ ]:
enc = tokenizer(build_input(EXPLAIN_TEXT, EXPLAIN_ASPECT),
                return_tensors='pt', truncation=True, padding=True).to(device)
input_ids     = enc['input_ids']
attention_mask = enc['attention_mask']

with torch.no_grad():
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    target_class = int(torch.argmax(logits, dim=-1).item())

lig = LayerIntegratedGradients(
    lambda ids, mask: model(input_ids=ids, attention_mask=mask).logits,
    model.base_model.embeddings
)

attr, delta = lig.attribute(
    inputs=input_ids,
    baselines=torch.zeros_like(input_ids),
    additional_forward_args=(attention_mask,),
    target=target_class,
    return_convergence_delta=True
)

token_scores = attr.sum(dim=-1).squeeze(0).detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=list(range(len(tokens))), y=token_scores, color='#4C78A8', ax=ax)
ax.set_xticks(list(range(len(tokens))))
ax.set_xticklabels(tokens, rotation=90)
ax.set_title('Integrated Gradients — target=' + label_map[target_class])
plt.tight_layout()
ig_path = os.path.join(XAI_OUT_DIR, 'integrated_gradients.png')
plt.savefig(ig_path, dpi=150)
plt.show()
print('Convergence delta:', float(delta.mean().item()))
print('Saved:', ig_path)

### 5.4 — Attention Weights (last layer, mean over heads)

In [ ]:
att_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, output_attentions=True).to(device)
att_model.eval()

enc = tokenizer(build_input(EXPLAIN_TEXT, EXPLAIN_ASPECT),
                return_tensors='pt', truncation=True, padding=True).to(device)

with torch.no_grad():
    out = att_model(**enc)

att    = out.attentions[-1].mean(dim=1).squeeze(0).detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'].squeeze(0))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(att, cmap='coolwarm', ax=ax)
ax.set_xticks(np.arange(len(tokens)) + 0.5)
ax.set_xticklabels(tokens, rotation=90)
ax.set_yticks(np.arange(len(tokens)) + 0.5)
ax.set_yticklabels(tokens, rotation=0)
ax.set_title('Last-layer Attention (mean over heads)')
plt.tight_layout()
att_path = os.path.join(XAI_OUT_DIR, 'attention_heatmap.png')
plt.savefig(att_path, dpi=150)
plt.show()
print('Saved:', att_path)

### 5.5 — Gradient-based Token Importance (Grad × Input)

In [ ]:
enc = tokenizer(build_input(EXPLAIN_TEXT, EXPLAIN_ASPECT),
                return_tensors='pt', truncation=True, padding=True).to(device)
input_ids      = enc['input_ids']
attention_mask = enc['attention_mask']

emb = model.get_input_embeddings()(input_ids)
emb.retain_grad()

logits = model(inputs_embeds=emb, attention_mask=attention_mask).logits
target = int(torch.argmax(logits, dim=-1).item())

model.zero_grad()
logits[0, target].backward()

token_importance = (emb.grad * emb).sum(dim=-1).squeeze(0).detach().cpu().numpy()
token_importance = np.maximum(token_importance, 0)
if token_importance.max() > 0:
    token_importance = token_importance / token_importance.max()

tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=list(range(len(tokens))), y=token_importance, color='#F58518', ax=ax)
ax.set_xticks(list(range(len(tokens))))
ax.set_xticklabels(tokens, rotation=90)
ax.set_ylim(0, 1.05)
ax.set_title('Gradient-based Token Importance — target=' + label_map[target])
plt.tight_layout()
grad_path = os.path.join(XAI_OUT_DIR, 'gradient_token_importance.png')
plt.savefig(grad_path, dpi=150)
plt.show()
print('Saved:', grad_path)

## Bước 6 — Tóm tắt kết quả

In [ ]:
import json

summary = {
    'model_used': MODEL_PATH,
    'separator': SEP,
    'device': str(device),
    'explain_text': EXPLAIN_TEXT,
    'explain_aspect': EXPLAIN_ASPECT,
    'predicted_label': label_map[pred],
    'probabilities': {label_map[i]: float(probs[i]) for i in range(3)},
    'xai_outputs': {
        'lime': lime_path,
        'integrated_gradients': ig_path,
        'attention_heatmap': att_path,
        'gradient_token_importance': grad_path,
    },
}

summary_path = os.path.join(XAI_OUT_DIR, 'xai_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('=== XAI Summary ===')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nAll outputs saved to:', XAI_OUT_DIR)